In [ ]:
raw = 24  # count
offset = 4  # count
scale = 0.05  # m/count
distance_m = (raw - offset) * scale
print(distance_m)

1.0


In [ ]:
class SensorCalibrator:
    def __init__(self, offset, scale):
        self.offset = offset
        self.scale = scale

    def apply(self, raw):
        return (raw - self.offset) * self.scale


def apply_named(registry, raw_by_name):
    return {name: registry[name].apply(raw) for name, raw in raw_by_name.items()}

In [ ]:
demo_registry = {"front": SensorCalibrator(4, 0.05)}
demo_result = apply_named(demo_registry, {"front": 24})
print(demo_result["front"])

1.0


In [ ]:
def record_frames(registry, frames):
    return [
        {"t_ms": frame["t_ms"], "values": apply_named(registry, frame["raw"])}
        for frame in frames
    ]


registry = {"front": SensorCalibrator(4, 0.05)}
frames = [
    {"t_ms": 0, "raw": {"front": 28}},
    {"t_ms": 100, "raw": {"front": 14}},
    {"t_ms": 200, "raw": {"front": 10}},
]
records = record_frames(registry, frames)
for item in records:
    print(item["t_ms"], round(item["values"]["front"], 2))

0 1.2
100 0.5
200 0.3


In [ ]:
def choose_action(front_m, stop_m):
    if front_m <= stop_m:
        return "STOP"
    else:
        return "GO"

In [ ]:
stop_m = 0.50
for distance_m in [0.25, 0.50, 0.75]:
    print(distance_m, choose_action(distance_m, stop_m))

0.25 STOP
0.5 STOP
0.75 GO


In [13]:
from copy import deepcopy


def replay(records, stop_m):
    return [
        {
            "t_ms": row["t_ms"],
            "front_m": row["values"]["front"],
            "action": choose_action(row["values"]["front"], stop_m),
        }
        for row in records
    ]


before = deepcopy(records)
for threshold in [0.50, 0.25]:
    print("stop_m =", threshold)
    result = replay(records, threshold)
    for row in result:
        print(
            row["t_ms"],
            round(row["front_m"], 2),
            row["action"],
        )

        print("-----")
assert records == before

stop_m = 0.5
0 1.2 GO
-----
100 0.5 STOP
-----
200 0.3 STOP
-----
stop_m = 0.25
0 1.2 GO
-----
100 0.5 GO
-----
200 0.3 GO
-----


### 관찰할 세가지

1. 한 기준 안에서 먼 거리와 가까운 거리의 판단이 어떻게 다른가?

- GO, STOP으로 다르다

2. 입력은 고정하고 기준만 바꾸면 어떤 시점의 판단이 달라지는가?

- 100, 200 시점의 판단이 달라짐

3. 마지막 assert가 통과하는가? 기준 실험 때문에 원본 기록이 바뀌면 안 돼요.

- ㅇㅇ
